### 1. Library

In [1]:
from pathlib import Path
from zipfile import ZipFile
import shutil


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

BRONZE_DIR = PROJECT_ROOT / "data" / "bronze"
STAGING_DIR = PROJECT_ROOT / "data" / "staging"

ano_teste = 2022

zip_origem = (
    BRONZE_DIR /
    f"microdados_enem_{ano_teste}.zip"
)

diretorio_destino = (
    STAGING_DIR /
    f"ano_{ano_teste}"
)

print(f"Origem : {zip_origem}")
print(f"Destino: {diretorio_destino}")

Origem : /home/akel/PycharmProjects/ENEM/data/bronze/microdados_enem_2022.zip
Destino: /home/akel/PycharmProjects/ENEM/data/staging/ano_2022


### 2.Identificar o CSV principal

In [2]:
with ZipFile(zip_origem, mode="r") as arquivo_zip:
    arquivos_csv = [
        info
        for info in arquivo_zip.infolist()
        if not info.is_dir()
        and info.filename.lower().endswith(".csv")
        and "MICRODADOS_ENEM" in Path(info.filename).name.upper()
    ]

if len(arquivos_csv) != 1:
    raise RuntimeError(
        f"Esperado 1 CSV principal, "
        f"mas foram encontrados {len(arquivos_csv)}."
    )

csv_principal = arquivos_csv[0]

print(f"Arquivo interno: {csv_principal.filename}")
print(
    f"Tamanho: "
    f"{csv_principal.file_size / 1024**2:.2f} MB"
)

Arquivo interno: DADOS/MICRODADOS_ENEM_2022.csv
Tamanho: 1495.51 MB


### 3.Verificar espaço disponível

In [3]:
espaco_livre = shutil.disk_usage(
    STAGING_DIR
).free

tamanho_necessario = csv_principal.file_size

print(
    f"Espaço disponível: "
    f"{espaco_livre / 1024**3:.2f} GB"
)

print(
    f"Espaço necessário: "
    f"{tamanho_necessario / 1024**3:.2f} GB"
)

if espaco_livre < tamanho_necessario * 1.2:
    raise OSError(
        "Espaço insuficiente para realizar "
        "a extração com margem de segurança."
    )

Espaço disponível: 14.68 GB
Espaço necessário: 1.46 GB


### 4 — extrair somente o CSV

In [4]:
diretorio_destino.mkdir(
    parents=True,
    exist_ok=True,
)

arquivo_destino = (
    diretorio_destino /
    Path(csv_principal.filename).name
)

if arquivo_destino.exists():
    raise FileExistsError(
        f"O arquivo já existe: {arquivo_destino}"
    )

with ZipFile(zip_origem, mode="r") as arquivo_zip:
    with arquivo_zip.open(
        csv_principal.filename,
        mode="r",
    ) as origem:
        with arquivo_destino.open("xb") as destino:
            shutil.copyfileobj(
                origem,
                destino,
                length=16 * 1024**2,
            )

print(f"Arquivo extraído: {arquivo_destino}")

Arquivo extraído: /home/akel/PycharmProjects/ENEM/data/staging/ano_2022/MICRODADOS_ENEM_2022.csv


### 5.Validar o tamanho

In [5]:
tamanho_extraido = arquivo_destino.stat().st_size
tamanho_esperado = csv_principal.file_size

print(
    f"Tamanho esperado: "
    f"{tamanho_esperado / 1024**2:.2f} MB"
)

print(
    f"Tamanho extraído: "
    f"{tamanho_extraido / 1024**2:.2f} MB"
)

if tamanho_extraido != tamanho_esperado:
    raise RuntimeError(
        "O tamanho extraído é diferente "
        "do tamanho registrado no ZIP."
    )

print("Extração validada com sucesso.")

Tamanho esperado: 1495.51 MB
Tamanho extraído: 1495.51 MB
Extração validada com sucesso.
